In [1]:
using CSV, DataFrames, Statistics
using PyPlot # plotting library
using PyCall
using Printf
using FStrings
using LaTeXStrings # for L"$x$" to work instead of needing to do "\$x\$"

# check out all of the styles! https://matplotlib.org/3.1.1/gallery/style_sheets/style_sheets_reference.html
# PyPlot.matplotlib.style.use("seaborn-pastel")

# change settings for all plots at once, e.g. font size
rcParams = PyPlot.PyDict(PyPlot.matplotlib."rcParams")
rcParams["font.size"] = 16

#make jupyter notebook print more dataframe columns
ENV["COLUMNS"]=160

# important attributes for this analysis
hand = "2v2"
subjects = ["sub1", "sub2", "sub3"]
t_labels = ["a", "b", "c", "d", "e", "f", "g", "h"]
rotations = ["n"]
translations = ['a','b','c','d','e','f','g','h']

metrics = [:dist, :arc_len, :area_btwn, :max_a_loc, :max_a_reg, :max_err, :mvt_eff, :t_fd, :r_fd]
metric_sds = [:dist_sd, :arc_len, :area_btwn_sd, :max_a_loc_sd, :max_a_reg_sd, :max_err_sd, :mvt_eff_sd, :t_fd_sd, :r_fd_sd]

metric_to_color = Dict(:dist=>"xkcd:tan", :arc_len=>"xkcd:light blue", :area_btwn=>"xkcd:dark red", :max_a_loc=>"xkcd:light green", :max_a_reg=>"xkcd:dark green", :max_err=>"xkcd:blue grey", :mvt_eff=>"xkcd:dark orange", :r_fd=>"xkcd:red brown", :t_fd=>"xkcd:brown")


print("Ready to go!")

Ready to go!

-----

# Reading in the Data
Now reading in the metrics data for 2v2.

In [2]:
df_hand = CSV.read(f"/home/cras/Workspace/asterisk_test/data/results/{hand}_trial_metrics.csv", DataFrame, copycols=true)
df_hand_avg = CSV.read(f"/home/cras/Workspace/asterisk_test/data/results/{hand}_trial_metrics_avgd.csv", DataFrame, copycols=true)
df_hand_std = CSV.read(f"/home/cras/Workspace/asterisk_test/data/results/{hand}_trial_metric_avgd_sds.csv", DataFrame, copycols=true)
first(df_hand, 23)

Row,trial,dist,t_fd,fd,max_err,max_rot_err,mvt_eff,arc_len,area_btwn,max_a_reg,max_a_loc
,String15,Float64,Float64,Float64,Int64,Int64,Float64,Float64,Float64,Int64,Int64
1,2v2_x_mm_s11_1,69.05,0.0567226,7.87706,-1,-1,0.863886,79.9296,0.0163064,-1,-1
2,2v2_x_mm_s11_2,75.387,0.0815,1.64746,-1,-1,0.887307,84.9616,0.0113604,-1,-1
3,2v2_x_mm_s11_3,65.358,0.0741,0.0741,-1,-1,0.953935,68.5141,0.0109033,-1,-1
4,2v2_x_mm_s12_1,72.042,0.141376,4.26271,-1,-1,0.324026,222.334,0.0826561,-1,-1
5,2v2_x_mm_s12_2,70.206,0.0945952,2.82698,-1,-1,0.524015,133.977,0.0602201,-1,-1
6,2v2_x_mm_s12_3,76.781,0.0556289,1.64717,-1,-1,0.698704,109.891,0.00962037,-1,-1
7,2v2_x_mm_s13_1,55.795,0.200087,2.1117,-1,-1,0.439003,127.095,0.0639047,-1,-1
8,2v2_x_mm_s13_2,57.696,0.1037,1.85328,-1,-1,0.544001,106.059,0.0123214,-1,-1
9,2v2_x_mm_s13_3,67.104,0.0913219,16.9582,-1,-1,0.788482,85.1053,0.0120396,-1,-1


In [3]:
# breaking out important info from the trial id for easier data access

function condition_df(df::DataFrame)
    subject_nums = []
    translation_labels = []
    numbers = []
    for row in eachrow(df)
        label = row[:trial]
        #sub1_2v2_a_n_1
    #     push!(subject_nums, x[4])
    #     push!(translation_labels,x[10])
    #     push!(numbers, x[14])
        x = split(label, "_")
        push!(subject_nums, x[1][4])
        push!(translation_labels, x[3][1])
        push!(numbers, x[5][1])
    end
    s = convert(Array{Char,1}, subject_nums)
    d = convert(Array{Char,1}, translation_labels)
    n = convert(Array{Char,1}, numbers)

    insertcols!(df, 2, :subj=>s)
    insertcols!(df, 3, :dir=>d)
    insertcols!(df, 4, :num=>n)
end

condition_df(df_hand)
condition_df(df_hand_avg)
condition_df(df_hand_std)

first(df_hand, 2)

LoadError: BoundsError: attempt to access 3-codeunit SubString{String15} at index [4]

  Julia 1.12 has introduced more strict world age semantics for global bindings.
  !!! This code may malfunction under Revise.
  !!! This code will error in future versions of Julia.
Hint: Add an appropriate `invokelatest` around the access to this binding.
To make this warning an error, and hence obtain a stack trace, use `julia --depwarn=error`.


## Asterisk Plot

![Avg Asterisk](./pics/avgd_2v2_3subs_n.jpg)

-----

# Histograms of Metrics
Now I will produce histogram plots of the metrics to see what kind of a distribution we get for all the trials.
Shamelessly taken from my previous datascience course at: [link](https://github.com/SimonEnsemble/data_science_course)

In [4]:

for m in metrics
    figure()
    hist(df_hand[:, m], color=metric_to_color[m]) #, normed=true)
    scatter(df_hand[:, m], [-1 for i = 1:nrow(df_hand)], marker="+", color="r")
    xlabel(f"{m}")
    ylabel("# Trials")
    title(f"Distribution of {m} values")
end

LoadError: ArgumentError: column name "r_fd" not found in the data frame; existing most similar names are: "t_fd" and "fd"

### Observations of Histogram Plots
Based on what I see, here are the observations per metric (in order of appearance top to down):
* **total distance**: values are distributed
* **arc length**: mostly concentrated at low arc lengths
* **max area region location**: mostly, the region of max error area is most commonly found at the end
* **max area region**: at the locations of max error area, the area of error (aka the area between the path and the target line) is still relatively low
* **max error**: values are distributed
* **movement efficiency**: values are distributed
* **rotational frechet distance**: aka max rotation error. Mostly low, but larger distribution at the low end
* translational frechet distance: all values are very low

----

# Metric Values by direction
Plot all metric values in bar charts by direction

In [5]:
# groups = groupby(df_hand, :dir)
# a_df = groups[1]
# b_df = groups[2]
# c_df = groups[3]
# d_df = groups[4]
# e_df = groups[5]
# f_df = groups[6]
# g_df = groups[7]
# h_df = groups[8]
# directions = [a_df, b_df, c_df, d_df, e_df, f_df, g_df, h_df]
# a_df

In [6]:
function compare_metrics(metric::Symbol, df::DataFrame) 
    dir_data = groupby(df, :dir)
    metric_data = []
    
    for dir in dir_data
        data = dir[!, metric]
        push!(metric_data, data)
    end
    plt_title = f"{hand}: {metric}"
    boxplot(metric_data)
    title(plt_title)
    xticks(1:8, t_labels)
    xlabel("Trial Directions")
end

println("Functions are ready!")

Functions are ready!


In [7]:
subjects = ["sub1", "sub2", "sub3"]
translations = ['a','b','c','d','e','f','g','h']
rotations = ["n"]

function get_trial_vals(df::DataFrame, metric::Symbol, translation::Char)
    desired_direction = filter(row -> row[:dir] == translation, df)
    names = desired_direction[!, :trial]
    values = desired_direction[!, metric]
    v = convert(Array{Float64,1}, values)  # just in case
    
    return v, names
end

function get_sd_vals(df::DataFrame, metric::Symbol, translation::Char)
    desired_direction = filter(row -> row[:dir] == translation, df)
    names = desired_direction[!, :trial]
    values = desired_direction[!, metric]
    v = convert(Array{Float64,1}, values)  # just in case
    
    return v, names
end

print("Functions are ready!")

Functions are ready!

-----

# Average Line Calculations

#### Dir: A
![Average Debug Plot for direction A](./pics/avgdebug_2v2_3subs_a_n.jpg)

#### Dir: B
![Average Debug Plot for direction B](./pics/avgdebug_2v2_3subs_b_n.jpg)

#### Dir: C
![Average Debug Plot for direction C](./pics/avgdebug_2v2_3subs_c_n.jpg)

#### Dir: D
![Average Debug Plot for direction D](./pics/avgdebug_2v2_3subs_d_n.jpg)

#### Dir: E
![Average Debug Plot for direction E](./pics/avgdebug_2v2_3subs_e_n.jpg)

#### Dir: F
![Average Debug Plot for direction F](./pics/avgdebug_2v2_3subs_f_n.jpg)

#### Dir: G
![Average Debug Plot for direction G](./pics/avgdebug_2v2_3subs_g_n.jpg)

#### Dir: H
![Average Debug Plot for direction H](./pics/avgdebug_2v2_3subs_h_n.jpg)


----

# Listing the Metric Values with standard deviations

In [8]:
first(df_hand_avg, 10)

Row,trial,dist,t_fd,fd,max_err,max_rot_err,mvt_eff,arc_len,area_btwn,max_a_reg,max_a_loc
,String,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,"2v2__n__pp__{'s12', 's11', 's13'}",62.8621,0.0761918,4.63869,-1.0,-1.0,0.642122,103.093,0.0180746,-1.0,-1.0
2,"2v2__n__mm__{'s12', 's11', 's13'}",68.7974,0.0796526,4.69776,-1.0,-1.0,0.751476,95.491,0.0189674,-1.0,-1.0


In [9]:
first(df_hand_std, 10)

Row,trial,dist,t_fd,fd,max_err,max_rot_err,mvt_eff,arc_len,area_btwn,max_a_reg,max_a_loc
,String,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,"2v2__n__pp__{'s12', 's11', 's13'}",6.42227,0.016375,5.57225,0.0,0.0,0.158405,22.9744,0.0129522,0.0,0.0
2,"2v2__n__mm__{'s12', 's11', 's13'}",5.94194,0.0172167,5.50334,0.0,0.0,0.156303,20.6634,0.0169508,0.0,0.0


In [10]:
function print_avg_and_std(metric::Symbol)
    translations = ['a','b','c','d','e','f','g','h']
    #translations = ['f']
    # made another metric list so it matches the order for the figure in the paper
    metric_list = [:dist, :arc_len, :mvt_eff, :max_err, :t_fd, :area_btwn, :max_a_reg, :max_a_loc, :r_fd]

    for i in range(1,9, step=1)
        println(" ")
        println(f"{metric_list[i]} metric")
        
        for t in translations
            vals, _ = get_trial_vals(df_hand_avg, metric_list[i], t)
            stds, _ = get_trial_vals(df_hand_std, metric_list[i], t)


            for i in range(1,length(vals), step=1)
                v = round(vals[i], digits=2)
                s = round(stds[i], digits=2)
                println(f"Dir {t}: {v} +/- {s}")

            end

#            print(t)
#            print(": ")
#            print(vals)
#            print(" | ")
#            println(stds)
        end
    end
end

print_avg_and_std(:t_fd)

 
dist metric


LoadError: ArgumentError: column name "dir" not found in the data frame; existing most similar names are: "dist"

  Julia 1.12 has introduced more strict world age semantics for global bindings.
  !!! This code may malfunction under Revise.
  !!! This code will error in future versions of Julia.
Hint: Add an appropriate `invokelatest` around the access to this binding.
To make this warning an error, and hence obtain a stack trace, use `julia --depwarn=error`.


# T Tests for Symmetry

In [11]:
using HypothesisTests

LoadError: ArgumentError: Package HypothesisTests not found in current path.
- Run `import Pkg; Pkg.add("HypothesisTests")` to install the HypothesisTests package.

In [12]:
function compare_2_directions(metric::Symbol)
    translations = ['a','b','c','d','e','f','g','h']
    results_report = zeros(8, 8)
    
    for t in translations
        i = findall(x->x==t, translations)[1]
        n = 0
        
        for t2 in translations[i:end]
            v1, _ = get_trial_vals(df_hand, metric, t)
            v2, _ = get_trial_vals(df_hand, metric, t2)
            
            result = pvalue(UnequalVarianceTTest(v1, v2))
            r = round(result, digits=4)
            results_report[i, i+n] = r
            
            n = n+1
        end
        println(" ")
    end

    r_df = DataFrame(trials=translations = ["a", "b", "c", "d", "e", "f", "g", "h"], 
                     a=results_report[1,:], b=results_report[2,:], c=results_report[3,:],
                     d=results_report[4,:], e=results_report[5,:], f=results_report[6,:],
                     g=results_report[7,:], h=results_report[8,:])
    
    return r_df
end

# temp = compare_2_directions(:arc_len)
println("Functions are ready!")

Functions are ready!


In [13]:
# for m in metrics
#     maxes = []
#     mins = []
#     r = compare_2_directions(m)
    
#     println(f"{m} data")
#     println("   ")
#     println(r)
#     println("================================================")
#     println("    ")
# end

----

# Data Analysis 

![Avg Asterisk](./pics/avgd_2v2_3subs_n.jpg)

**Hypotheses:**
* df

### Total Distance Travelled


In [14]:
compare_metrics(:dist, df_hand)
compare_2_directions(:dist)

LoadError: ArgumentError: column name "dir" not found in the data frame; existing most similar names are: "dist"

### Arc Length

In [15]:
compare_metrics(:arc_len, df_hand)
compare_2_directions(:arc_len)

LoadError: ArgumentError: column name "dir" not found in the data frame; existing most similar names are: "dist"

### Movement Efficiency
total_distance / arc_length

In [16]:
compare_metrics(:mvt_eff, df_hand)
compare_2_directions(:mvt_eff)

LoadError: ArgumentError: column name "dir" not found in the data frame; existing most similar names are: "dist"

### Max Error

In [17]:
compare_metrics(:max_err, df_hand)
compare_2_directions(:max_err)

LoadError: ArgumentError: column name "dir" not found in the data frame; existing most similar names are: "dist"

### Translational Frechet Distance

In [18]:
compare_metrics(:t_fd, df_hand)
compare_2_directions(:t_fd)

LoadError: ArgumentError: column name "dir" not found in the data frame; existing most similar names are: "dist"

### Max Rotation Error / Rotational Frechet Distance

In [19]:
compare_metrics(:r_fd, df_hand)
compare_2_directions(:r_fd)

LoadError: ArgumentError: column name "dir" not found in the data frame; existing most similar names are: "dist"

### Total Area Between Curves
Between object path and target line

In [20]:
compare_metrics(:area_btwn, df_hand)
compare_2_directions(:area_btwn)

LoadError: ArgumentError: column name "dir" not found in the data frame; existing most similar names are: "dist"

### Max Area Region
The area between the curves of a scrolling window of about 20% total_distance width.

In [21]:
compare_metrics(:max_a_reg, df_hand)
compare_2_directions(:max_a_reg)

LoadError: ArgumentError: column name "dir" not found in the data frame; existing most similar names are: "dist"

### Max Area Location
The location of the center of the max area region, represented as a percentage along the full target line (that is, some percentage of 0.5).

In [22]:
compare_metrics(:max_a_loc, df_hand)
compare_2_directions(:max_a_loc)

LoadError: ArgumentError: column name "dir" not found in the data frame; existing most similar names are: "dist"

-----

## Comparing PValues Directly, Direction to Direction, all Metrics

In [23]:
translations = ['a','b','c','d','e','f','g','h']
results_report = zeros(8, 8)

for t in translations
    i = findall(x->x==t, translations)[1]
    n = 0

    for t2 in translations[(i+1):end]
        println(f"{t} v {t2}")
        for m in metrics
            v1, _ = get_trial_vals(df_hand, m, t)
            v2, _ = get_trial_vals(df_hand, m, t2)

            result = pvalue(UnequalVarianceTTest(v1, v2))
            r = round(result, digits=4)
            println(f"{m} => {r}")
        end

        println("  ")
    end
    println("====================")
end

a v b


LoadError: ArgumentError: column name "dir" not found in the data frame; existing most similar names are: "dist"

In [24]:
function aggregate_pvalues()
    translations = ['a','b','c','d','e','f','g','h']
    results_report = zeros(8, 8)
    
    for t in translations
        i = findall(x->x==t, translations)[1]
        n = 0
        
        for t2 in translations[i:end]
            total_sigs = 0
            
            for m in metrics
                v1, _ = get_trial_vals(df_hand, m, t)
                v2, _ = get_trial_vals(df_hand, m, t2)

                result = pvalue(UnequalVarianceTTest(v1, v2))
                
                if result <= 0.05
                    total_sigs = total_sigs + 1
                end
                
                if t == t2
                    total_sigs = NaN
                end
            end
            perc_sigs = total_sigs / length(metrics)
            results_report[i, i+n] = perc_sigs
            
            n = n+1
        end
        println(" ")
    end

    r_df = DataFrame(trials=translations = ["a", "b", "c", "d", "e", "f", "g", "h"], 
                     a=results_report[1,:], b=results_report[2,:], c=results_report[3,:],
                     d=results_report[4,:], e=results_report[5,:], f=results_report[6,:],
                     g=results_report[7,:], h=results_report[8,:])
    
    return r_df
end

aggregate_results = aggregate_pvalues()
# CSV.write(aggregate_results)
aggregate_results

LoadError: ArgumentError: column name "dir" not found in the data frame; existing most similar names are: "dist"

  Julia 1.12 has introduced more strict world age semantics for global bindings.
  !!! This code may malfunction under Revise.
  !!! This code will error in future versions of Julia.
Hint: Add an appropriate `invokelatest` around the access to this binding.
To make this warning an error, and hence obtain a stack trace, use `julia --depwarn=error`.


In [25]:
colormap = PyPlot.matplotlib.cm.get_cmap("bwr")
colors = colormap.(reverse(range(0.1, stop=0.9, length=10)))

10-element Vector{NTuple{4, Float64}}:
 (1.0, 0.196078431372549, 0.196078431372549, 1.0)
 (1.0, 0.3764705882352941, 0.3764705882352941, 1.0)
 (1.0, 0.5568627450980392, 0.5568627450980392, 1.0)
 (1.0, 0.7294117647058824, 0.7294117647058824, 1.0)
 (1.0, 0.9098039215686274, 0.9098039215686274, 1.0)
 (0.9098039215686274, 0.9098039215686274, 1.0, 1.0)
 (0.7294117647058823, 0.7294117647058823, 1.0, 1.0)
 (0.5568627450980392, 0.5568627450980392, 1.0, 1.0)
 (0.3764705882352941, 0.3764705882352941, 1.0, 1.0)
 (0.19607843137254902, 0.19607843137254902, 1.0, 1.0)

In [26]:
# make a nice table with color coding, highlighting directions that have and don't have significant differences from each other

fig, ax = subplots()

# Hide axes because there isn't a plot to go with it
ax.axis("off")

results_table = convert(Matrix, aggregate_results)
# the_table = table(results_table, colLabels=names(aggregate_results))

# the_table.get_celld()[(1,0)].set_facecolor("#56b5fd")

# colormap = PyPlot.matplotlib.cm.get_cmap("bwr")

# for i in range(1, stop=8)
#     for j in range(1, stop=9)
#         val = aggregate_results[i,j]
        
#         if typeof(val) == String
#             c = "xkcd:grey blue"
#         elseif isnan(val)
#             c = "xkcd:dark grey"
#         else
#             c = colormap(val)
#         end
        
#         the_table.get_celld()[(i-1,j-1)].set_facecolor(c)
#     end
# end


LoadError: UndefVarError: `aggregate_results` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.

![Avg Asterisk](./pics/avgd_2v2_3subs_n.jpg)

# Clustering for Symmetry

In [27]:
using Clustering

# use kmeans or hierarchical clustering single linkage?

LoadError: ArgumentError: Package Clustering not found in current path.
- Run `import Pkg; Pkg.add("Clustering")` to install the Clustering package.

In [28]:
test = [1, 2, 3, 4, 5]
t = 2
i = findall(x->x!=t, test)

4-element Vector{Int64}:
 1
 3
 4
 5